# 兩階段訓練 — 缺陷偵測 + 模式分類
**Stage 1**：二分類（none / non-none）

先讓模型學習辨識「有無缺陷」。調低推論閾值（`S1_THRESHOLD`）以提高 non-none Recall，
確保 Edge-Loc、Loc、Scratch 等較難辨識的類別不被誤判為 none。

**Stage 2**：多類別分類（8 種缺陷模式）

對 Stage 1 判定為 non-none 的樣本，再辨識具體的缺陷模式類別。

Cell 執行順序：
- **Cell 1**：環境設定（只需執行一次）
- **Cell 2**：實驗設定（換參數只改這裡，改完重跑 Cell 4+）
- **Cell 3**：載入資料（只需執行一次，換設定不需重跑）
- **Cell 4**：Stage 1 — 準備資料集與模型
- **Cell 5**：Stage 1 — 訓練（含即時圖表）
- **Cell 6**：Stage 1 — 測試集評估與圖表
- **Cell 7**：Stage 2 — 準備資料集與模型
- **Cell 8**：Stage 2 — 訓練（含即時圖表）
- **Cell 9**：Stage 2 — 測試集評估與圖表
- **Cell 10**：儲存結果（選用）

In [ ]:
import sys
sys.path.insert(0, '..')
# Cell 1 — 環境設定（只需執行一次）
%matplotlib inline
import copy
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import clear_output
from tabulate import tabulate
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

from engine.trainer import find_best_f1, get_class_weights, print_metrics, print_per_class_f1, run_one_epoch, train_model
from engine.visualize import plot_cm, plot_history
from models.builder import build_model
from models.inference import predict_two_stage
from data.dataset import WaferDataset, extract_labeled_patterned_data, load_raw_data, make_dataloaders, split_df
from constant import LABEL_MAP, NONE_NONONE_LABEL_MAP, PATTERN_LABEL_MAP
from utils import make_output_dir, save_results

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"使用裝置：{device}")

In [ ]:
# Cell 2 — 實驗設定（換參數只改這裡，改完重跑 Cell 4+）
EXPERIMENT_NAME = "two_stage"
IMAGE_SIZE      = (64,64)
RANDOM_SEED     = 42

# Stage 1：缺陷偵測（none / non-none 二分類）
S1_BATCH_SIZE  = 32
S1_EPOCHS      = 24
S1_LR          = 1e-3
S1_THRESHOLD   = 0.3   # 調低閾值讓 non-none Recall 更高

# Stage 2：缺陷模式分類（8 類）
S2_BATCH_SIZE  = 32
S2_EPOCHS      = 24
S2_LR          = 1e-3

In [ ]:
# Cell 3 — 載入資料（只需執行一次，換設定不用重跑）
print("下載/讀取 WM-811K 資料集...")
df = load_raw_data()
df_with_label, df_with_pattern, _ = extract_labeled_patterned_data(df)
print(f"有標籤資料：{len(df_with_label)} 筆")
print(f"有缺陷模式資料（non-none）：{len(df_with_pattern)} 筆")

## Stage 1：缺陷偵測（none / non-none 二分類）

In [ ]:
# Cell 4 — Stage 1：準備資料集與模型
df_s1 = copy.deepcopy(df_with_label)
df_s1["hasDefect"] = (df_s1["failureType"] != "none").astype(int)
# 0 = none，1 = non-none

train_s1, val_s1, test_s1 = split_df(
    df_s1, "hasDefect", random_seed=RANDOM_SEED,
)

num_classes_s1 = 2
datasets_s1 = {
    "train": WaferDataset(train_s1, label_col="hasDefect", label_map=None, image_size=IMAGE_SIZE),
    "val":   WaferDataset(val_s1,   label_col="hasDefect", label_map=None, image_size=IMAGE_SIZE),
    "test":  WaferDataset(test_s1,  label_col="hasDefect", label_map=None, image_size=IMAGE_SIZE),
}
dataloaders_s1 = make_dataloaders(datasets_s1, S1_BATCH_SIZE)

criterion_s1 = nn.CrossEntropyLoss()
model_s1 = build_model(num_classes_s1, device)
params_s1 = filter(lambda p: p.requires_grad, model_s1.parameters())
optimizer_s1 = AdamW(params_s1, lr=S1_LR)
scheduler_s1 = ReduceLROnPlateau(optimizer_s1, mode="max", patience=2, factor=0.5)
print(f"模型與資料集準備完成，類別數：{num_classes_s1}")
print(f"Stage 1 推論閾值：{S1_THRESHOLD}")

In [ ]:
# Cell 5 — Stage 1：訓練
print("===== Stage 1: Defect Detection (none / non-none) =====")
best_s1, best_recall_s1, best_cm_s1, best_f1pc_s1, history_s1 = train_model(
    model_s1, device, criterion_s1, optimizer_s1, scheduler_s1, dataloaders_s1,
    learning_depend_metric="recall",
    threshold=S1_THRESHOLD,
    num_classes=num_classes_s1,
    epochs=S1_EPOCHS,
    lr=S1_LR,
    early_stopping=True, patience=6,
    class_names=["none", "non-none"],
)
best_f1_s1 = find_best_f1("val_recall", best_recall_s1, history_s1)
print(f"\nStage 1 完成，最佳驗證 Recall：{best_recall_s1:.4f}，對應 F1：{best_f1_s1:.4f}")

In [ ]:
# Cell 6 — Stage 1：測試集評估與圖表
model_s1.load_state_dict(best_s1)
test_m_s1 = run_one_epoch(
    model_s1, device, dataloaders_s1["test"], criterion_s1,
    num_classes=num_classes_s1, threshold=S1_THRESHOLD,
)
print(f"\n--- Stage 1 Test Results (threshold={S1_THRESHOLD}) ---")
print(tabulate(
    [[f"{test_m_s1['loss']:.4f}", f"{test_m_s1['f1']:.4f}",
      f"{test_m_s1['acc']:.4f}", f"{test_m_s1['precision']:.4f}", f"{test_m_s1['recall']:.4f}"]],
    headers=["Loss", "F1", "Acc", "Precision", "Recall"],
    tablefmt="grid",
))
plot_history(history_s1, show=True)
plot_cm(best_cm_s1, num_classes_s1, show=True)

## Stage 2：缺陷模式分類（8 類）

In [ ]:
# Cell 7 — Stage 2：準備資料集與模型
num_classes_s2 = len(PATTERN_LABEL_MAP)  # 8

train_s2, val_s2, test_s2 = split_df(
    df_with_pattern, "failureType", random_seed=RANDOM_SEED,
)

datasets_s2 = {
    "train": WaferDataset(train_s2, label_map=PATTERN_LABEL_MAP, image_size=IMAGE_SIZE),
    "val":   WaferDataset(val_s2,   label_map=PATTERN_LABEL_MAP, image_size=IMAGE_SIZE),
    "test":  WaferDataset(test_s2,  label_map=PATTERN_LABEL_MAP, image_size=IMAGE_SIZE),
}
dataloaders_s2 = make_dataloaders(datasets_s2, S2_BATCH_SIZE)

criterion_s2 = nn.CrossEntropyLoss()
model_s2 = build_model(num_classes_s2, device)
params_s2 = filter(lambda p: p.requires_grad, model_s2.parameters())
optimizer_s2 = AdamW(params_s2, lr=S2_LR)
scheduler_s2 = ReduceLROnPlateau(optimizer_s2, mode="max", patience=2, factor=0.5)
print(f"模型與資料集準備完成，類別數：{num_classes_s2}")
print("Stage 2 資料集與模型準備完成")

In [ ]:
# Cell 8 — Stage 2：訓練
print("===== Stage 2: Defect Pattern Classification (8 classes) =====")
best_s2, best_f1_s2, best_cm_s2, best_f1pc_s2, history_s2 = train_model(
    model_s2, device, criterion_s2, optimizer_s2, scheduler_s2, dataloaders_s2,
    num_classes=num_classes_s2,
    epochs=S2_EPOCHS,
    lr=S2_LR,
    early_stopping=True, patience=3,
    class_names=list(PATTERN_LABEL_MAP.keys()),
)
print(f"\nStage 2 完成，最佳驗證 F1：{best_f1_s2:.4f}")

In [ ]:
# Cell 9 — Stage 2：測試集評估與圖表
model_s2.load_state_dict(best_s2)
test_m_s2 = run_one_epoch(model_s2, device, dataloaders_s2["test"], criterion_s2, num_classes=num_classes_s2)
print("\n--- Stage 2 Test Results ---")
print(tabulate(
    [[f"{test_m_s2['loss']:.4f}", f"{test_m_s2['f1']:.4f}",
      f"{test_m_s2['acc']:.4f}", f"{test_m_s2['precision']:.4f}", f"{test_m_s2['recall']:.4f}"]],
    headers=["Loss", "F1", "Acc", "Precision", "Recall"],
    tablefmt="grid",
))
print("\n--- Stage 2 Per-Class F1 ---")
print_per_class_f1(best_f1pc_s2, class_names=list(PATTERN_LABEL_MAP.keys()))
plot_history(history_s2, show=True)
plot_cm(best_cm_s2, num_classes_s2, show=True)

In [ ]:
# Cell 10 — 儲存結果（選用）
out_dir    = make_output_dir(EXPERIMENT_NAME, base="../outputs")
out_dir_s1 = out_dir / "stage1"
out_dir_s2 = out_dir / "stage2"
out_dir_s1.mkdir(parents=True, exist_ok=True)
out_dir_s2.mkdir(parents=True, exist_ok=True)


save_results(
    out_dir_s1, model_s1, best_s1, best_f1_s1, best_cm_s1, best_f1pc_s1,
    history_s1, num_classes_s1, NONE_NONONE_LABEL_MAP, IMAGE_SIZE,
)
save_results(
    out_dir_s2, model_s2, best_s2, best_f1_s2, best_cm_s2, best_f1pc_s2,
    history_s2, num_classes_s2, PATTERN_LABEL_MAP, IMAGE_SIZE,
)
print(f"結果已儲存至：{out_dir}")

In [ ]:
from explain.core import load_model
from explain.visualize import explain_samples
from pytorch_grad_cam import GradCAM

model_s1, metadata_s1 = load_model(f"../outputs/two_stage_20260531_110545/stage1")
model_s2, metadata_s2 = load_model(f"../outputs/two_stage_20260531_110545/stage2")

print(metadata_s1)
print(metadata_s2)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.data import DataLoader


dataset_unified = WaferDataset(test_s1, label_col="failureType", label_map=LABEL_MAP, image_size=IMAGE_SIZE)
loader_unified  = DataLoader(dataset_unified, batch_size=S1_BATCH_SIZE, shuffle=False, num_workers=2)

model_s1.eval()
model_s2.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in loader_unified:
        images = images.to(device, non_blocking=True)
        B = images.size(0)

        # Stage 1
        probs_s1 = torch.softmax(model_s1(images), dim=1)[:, 1]
        
        # probs_s1資料在GPU，後面final_preds[is_non_none]資料要去CPU，所以probs_s1也要去CPU
        is_non_none = (probs_s1 > S1_THRESHOLD).cpu()  # bool mask

        # Default: predict "none" = indesample_input =x 8
        final_preds = torch.full((B,), fill_value=8, dtype=torch.long)

        if is_non_none.any():
            s2_logits = model_s2(images[is_non_none])
            final_preds[is_non_none] = s2_logits.argmax(dim=1).cpu()

        all_preds.append(final_preds)
        all_labels.append(labels)

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

unified_metrics = {
        "acc":          accuracy_score(all_labels, all_preds),
        "precision":    precision_score(all_labels, all_preds, average="macro", zero_division=0),
        "recall":       recall_score(all_labels, all_preds, average="macro", zero_division=0),
        "f1":           f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "per_class_f1": f1_score(all_labels, all_preds, average=None, zero_division=0),
        "cm":           confusion_matrix(all_labels, all_preds, labels=list(range(9))),
    }

In [ ]:
NINE_CLASS_NAMES = list(LABEL_MAP.keys())

print("=" * 60)
print("兩階段 Cascade — 統一 9 類別測試結果")
print(tabulate(
  [[f"{unified_metrics['f1']:.4f}",
      f"{unified_metrics['acc']:.4f}", f"{unified_metrics['precision']:.4f}", f"{unified_metrics['recall']:.4f}"]], 
      headers=["F1", "Acc", "Precision", "Recall"], 
      tablefmt="grid"
      ))
print_per_class_f1(unified_metrics["per_class_f1"], class_names=NINE_CLASS_NAMES)
plot_cm(unified_metrics["cm"], 9)

In [ ]:
# 單筆推論時間

import time

from models.inference import predict_two_stage
from explain.core import preprocess_wafer
# 量測單樣本推論時間
sample_input = test_s1.iloc[0]

preprocessed_wafer, _ = preprocess_wafer(sample_input["waferMap"], IMAGE_SIZE, device)

with torch.no_grad():  # warmup
    _ = predict_two_stage(model_s1, model_s2, preprocessed_wafer, S1_THRESHOLD)
if device.type == "cuda":
    torch.cuda.synchronize()
elif device.type == "mps":
    torch.mps.synchronize()

N = 100
with torch.no_grad():
    t0 = time.perf_counter()
    for _ in range(N):
        _ = predict_two_stage(model_s1, model_s2, preprocessed_wafer, S1_THRESHOLD)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()
    t1 = time.perf_counter()

inference_time = (t1 - t0) / N * 1000  # ms per sample
print(f"單樣本推論時間（{N} 次平均）：{inference_time:.3f} ms")

**two_stage_20260531_110545** 訓練筆記：

發現：
- Stage 2 從第 4 epoch 之後 loss 不再下降、f1 不再上升，但到了 epoch 22 才提早停止，是否耐心訂的太多。
- val loss 有過擬合情況。epoch 5 val loss: 0.27 -> 表現最好的 epoch 16 val loss: 0.37。

**two_stage_20260606_xxx** 訓練筆記：

改動：
- 訓練 patience=3